# Project 2: Customer Churn & Revenue Leakage Analysis

**Question:** Which customers leave, how much revenue do we lose when they do, and which retention campaign is worth paying for?

**Stack:** SQL (SQLite) · Python (pandas, scikit-learn, matplotlib) · Excel · Power BI

**Note:** the data is synthetic and uses the same columns as Kaggle's `Churn_Modelling.csv`. To use the real file, drop `RowNumber` and `Surname`, save it as `data/churn.csv`, and skip the data-creation cell.

## 0. Setup
Run this notebook **top to bottom** (Run All). It creates the folders `data/`, `sql/` and `outputs/` next to the notebook.

First install the libraries (only needed once):

In [ ]:
# !pip install pandas numpy scikit-learn matplotlib openpyxl

In [ ]:
import sys, json, re, sqlite3
from pathlib import Path
import numpy as np
import pandas as pd
for d in ['data', 'sql', 'outputs/charts', 'outputs/powerbi', 'outputs/sql_results']:
    Path(d).mkdir(parents=True, exist_ok=True)
print('Working folder:', Path.cwd())

## 1. Create the dataset
Synthetic data with realistic patterns, saved to `data/churn.csv`. To use real Kaggle data instead, skip this cell and put your file in `data/` (see the project README for the column mapping).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 7
N = 10_000
OUT = Path.cwd() / "data" / "churn.csv"
rng = np.random.default_rng(SEED)

geography = rng.choice(["France", "Germany", "Spain"], N, p=[.50, .25, .25])
gender = rng.choice(["Female", "Male"], N, p=[.45, .55])
age = np.clip(18 + rng.gamma(shape=9, scale=2.4, size=N), 18, 92).round().astype(int)
tenure = rng.integers(0, 11, N)
credit = np.clip(rng.normal(650, 97, N), 350, 850).round().astype(int)

# Balance: German customers almost always hold a balance; ~half of France/Spain hold none
has_balance = np.where(geography == "Germany", rng.random(N) < 0.99, rng.random(N) < 0.50)
balance = np.where(has_balance, np.clip(rng.normal(120_000, 30_000, N), 5_000, 250_000), 0).round(2)

products = rng.choice([1, 2, 3, 4], N, p=[.51, .46, .027, .003])
has_card = (rng.random(N) < 0.70).astype(int)
active = (rng.random(N) < 0.52).astype(int)
salary = rng.uniform(1_000, 200_000, N).round(2)

# --- churn probability: patterns widely documented for retail-bank churn ---------------
product_effect = pd.Series(products).map({1: 0.0, 2: -1.6, 3: 2.6, 4: 5.0}).to_numpy()
logit = (
    -1.30
    + 0.075 * (np.minimum(age, 65) - 38)
    + 0.85 * (geography == "Germany")
    + 0.05 * (geography == "Spain")
    + 0.38 * (gender == "Female")
    - 1.05 * active
    + product_effect
    + 0.0000016 * (balance - 60_000)
    - 0.0007 * (credit - 650)
)
p = 1 / (1 + np.exp(-logit))
exited = (rng.random(N) < p).astype(int)

df = pd.DataFrame({
    "CustomerId": np.arange(15_600_000, 15_600_000 + N),
    "CreditScore": credit, "Geography": geography, "Gender": gender, "Age": age,
    "Tenure": tenure, "Balance": balance, "NumOfProducts": products, "HasCrCard": has_card,
    "IsActiveMember": active, "EstimatedSalary": salary, "Exited": exited,
})
OUT.parent.mkdir(exist_ok=True)
df.to_csv(OUT, index=False)
print(f"Saved {len(df):,} rows -> {OUT}")
print(f"Overall churn rate: {exited.mean():.1%}")

## 2. SQL analysis
The 11 queries live in one text block. They are saved to `sql/queries.sql`, loaded into an in-memory-style SQLite database, and run one by one. The same SQL works in PostgreSQL/MySQL with tiny changes.

In [ ]:
SQL_TEXT = r"""
-- =====================================================================================
-- PROJECT 2: CUSTOMER CHURN & REVENUE LEAKAGE  |  SQL ANALYSIS
-- Dialect: SQLite (standard SQL - works in PostgreSQL / MySQL 8+ with tiny changes).
--
-- `customers_v` is a VIEW over the raw `customers` table that adds:
--     is_churn       = Exited (1 = customer left)
--     age_band, balance_band, activity   (CASE WHEN bucketing)
--     annual_revenue = 2% net interest margin on balance + $50 fee income per product
--                      (ASSUMPTION - change it in run_sql.py and pipeline.py together)
-- =====================================================================================


-- name: 01_kpis
-- question: How big is the churn problem, in customers AND in dollars?
SELECT
    COUNT(*)                                              AS customers,
    SUM(is_churn)                                         AS churned,
    ROUND(100.0 * AVG(is_churn), 2)                       AS churn_rate_pct,
    ROUND(SUM(annual_revenue) / 1e6, 2)                   AS total_annual_revenue_mn,
    ROUND(SUM(CASE WHEN is_churn = 1 THEN annual_revenue END) / 1e6, 2) AS revenue_lost_mn,
    ROUND(100.0 * SUM(CASE WHEN is_churn = 1 THEN annual_revenue END) / SUM(annual_revenue), 2) AS revenue_lost_pct,
    ROUND(AVG(Tenure), 2)                                 AS avg_tenure_yrs
FROM customers_v;


-- name: 02_churn_by_geography
-- question: Which country loses the most customers and revenue?
SELECT
    Geography,
    COUNT(*)                                              AS customers,
    ROUND(100.0 * AVG(is_churn), 2)                       AS churn_rate_pct,
    ROUND(SUM(CASE WHEN is_churn = 1 THEN annual_revenue END) / 1e3, 1) AS revenue_lost_k
FROM customers_v
GROUP BY Geography
ORDER BY churn_rate_pct DESC;


-- name: 03_churn_by_products
-- question: Is the number of products a bank customer holds linked to churn?
SELECT
    NumOfProducts,
    COUNT(*)                                              AS customers,
    ROUND(100.0 * AVG(is_churn), 2)                       AS churn_rate_pct,
    ROUND(SUM(CASE WHEN is_churn = 1 THEN annual_revenue END) / 1e3, 1) AS revenue_lost_k
FROM customers_v
GROUP BY NumOfProducts
ORDER BY NumOfProducts;


-- name: 04_activity_by_products_matrix
-- question: Do inactive customers with a single product churn the most? (two-way matrix)
SELECT
    activity,
    ROUND(100.0 * AVG(CASE WHEN NumOfProducts = 1 THEN is_churn END), 2) AS churn_pct_1_product,
    ROUND(100.0 * AVG(CASE WHEN NumOfProducts = 2 THEN is_churn END), 2) AS churn_pct_2_products,
    ROUND(100.0 * AVG(CASE WHEN NumOfProducts >= 3 THEN is_churn END), 2) AS churn_pct_3plus_products,
    COUNT(*)                                                             AS customers
FROM customers_v
GROUP BY activity;


-- name: 05_churn_by_age_band
-- question: Which age groups churn? (CASE WHEN banding, with gap vs the overall rate)
SELECT
    age_band,
    COUNT(*)                                              AS customers,
    ROUND(100.0 * AVG(is_churn), 2)                       AS churn_rate_pct,
    ROUND(100.0 * (AVG(is_churn) - (SELECT AVG(is_churn) FROM customers_v)), 2) AS gap_vs_overall_pp
FROM customers_v
GROUP BY age_band
ORDER BY age_band;


-- name: 06_segment_revenue_leakage_ranked
-- question: Which balance x activity segments leak the most revenue? (RANK window function)
WITH seg AS (
    SELECT
        balance_band || ' | ' || activity                       AS segment,
        COUNT(*)                                                AS customers,
        AVG(is_churn)                                           AS churn_rate,
        SUM(CASE WHEN is_churn = 1 THEN annual_revenue END)     AS revenue_lost
    FROM customers_v
    GROUP BY balance_band, activity
)
SELECT
    segment,
    customers,
    ROUND(100 * churn_rate, 2)                                  AS churn_rate_pct,
    ROUND(revenue_lost / 1e3, 1)                                AS revenue_lost_k,
    RANK() OVER (ORDER BY revenue_lost DESC)                    AS leakage_rank
FROM seg
ORDER BY leakage_rank;


-- name: 07_pareto_revenue_loss
-- question: Do a few segments cause most of the loss? (cumulative share with SUM() OVER)
WITH seg AS (
    SELECT
        Geography || ' | ' || activity                          AS segment,
        SUM(CASE WHEN is_churn = 1 THEN annual_revenue ELSE 0 END) AS revenue_lost
    FROM customers_v
    GROUP BY Geography, activity
)
SELECT
    segment,
    ROUND(revenue_lost / 1e3, 1)                                AS revenue_lost_k,
    ROUND(100.0 * revenue_lost / SUM(revenue_lost) OVER (), 1)  AS share_pct,
    ROUND(100.0 * SUM(revenue_lost) OVER (ORDER BY revenue_lost DESC)
          / SUM(revenue_lost) OVER (), 1)                       AS cumulative_share_pct
FROM seg
ORDER BY revenue_lost DESC;


-- name: 08_tenure_churn
-- question: Do new customers leave faster than loyal ones?
SELECT
    Tenure                                                AS tenure_years,
    COUNT(*)                                              AS customers,
    ROUND(100.0 * AVG(is_churn), 2)                       AS churn_rate_pct,
    ROUND(100.0 * (AVG(is_churn) - AVG(AVG(is_churn)) OVER ()), 2) AS gap_vs_avg_of_years_pp
FROM customers_v
GROUP BY Tenure
ORDER BY Tenure;


-- name: 09_balance_quartiles
-- question: Among customers WITH a balance, do bigger balances churn more? (NTILE quartiles)
WITH q AS (
    SELECT is_churn, Balance, annual_revenue,
           NTILE(4) OVER (ORDER BY Balance) AS balance_quartile
    FROM customers_v
    WHERE Balance > 0
)
SELECT
    balance_quartile,
    ROUND(MIN(Balance), 0)                                AS min_balance,
    ROUND(MAX(Balance), 0)                                AS max_balance,
    COUNT(*)                                              AS customers,
    ROUND(100.0 * AVG(is_churn), 2)                       AS churn_rate_pct
FROM q
GROUP BY balance_quartile
ORDER BY balance_quartile;


-- name: 10_high_risk_profile
-- question: How concentrated is churn in a simple 'high-risk profile'? (CTE + flag)
-- profile = age 45+ AND inactive AND (Germany OR 3+ products OR single product)
WITH flagged AS (
    SELECT *,
        CASE WHEN Age >= 45 AND IsActiveMember = 0
                  AND (Geography = 'Germany' OR NumOfProducts >= 3 OR NumOfProducts = 1)
             THEN 'High-risk profile' ELSE 'Everyone else' END AS profile
    FROM customers_v
)
SELECT
    profile,
    COUNT(*)                                              AS customers,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)    AS pct_of_customers,
    ROUND(100.0 * AVG(is_churn), 2)                       AS churn_rate_pct,
    ROUND(100.0 * SUM(is_churn) / SUM(SUM(is_churn)) OVER (), 1) AS pct_of_all_churners
FROM flagged
GROUP BY profile;


-- name: 11_top_valuable_churners_per_country
-- question: Who are the 3 most valuable lost customers in each country? (ROW_NUMBER + PARTITION BY)
WITH ranked AS (
    SELECT
        CustomerId, Geography, Age, NumOfProducts,
        ROUND(Balance, 0)        AS balance,
        ROUND(annual_revenue, 0) AS annual_revenue,
        ROW_NUMBER() OVER (PARTITION BY Geography ORDER BY annual_revenue DESC) AS rn
    FROM customers_v
    WHERE is_churn = 1
)
SELECT Geography, CustomerId, Age, NumOfProducts, balance, annual_revenue
FROM ranked
WHERE rn <= 3
ORDER BY Geography, annual_revenue DESC;
"""
(Path("sql") / "queries.sql").write_text(SQL_TEXT)

In [ ]:
import re
import sqlite3
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
DB_PATH = ROOT / "data" / "churn.db"
OUT_DIR = ROOT / "outputs" / "sql_results"

# ASSUMPTION: revenue per customer = 2% net interest margin on balance + $50 fees per product.
# Keep these in sync with NIM_RATE / FEE_PER_PRODUCT in pipeline.py.
VIEW_SQL = """
DROP VIEW IF EXISTS customers_v;
CREATE VIEW customers_v AS
SELECT
    *,
    Exited AS is_churn,
    CASE WHEN Age < 30 THEN '1) <30' WHEN Age < 40 THEN '2) 30-39' WHEN Age < 50 THEN '3) 40-49'
         WHEN Age < 60 THEN '4) 50-59' ELSE '5) 60+' END                     AS age_band,
    CASE WHEN Balance = 0 THEN '1) No balance' WHEN Balance < 100000 THEN '2) <100k'
         ELSE '3) 100k+' END                                                 AS balance_band,
    CASE WHEN IsActiveMember = 1 THEN 'Active' ELSE 'Inactive' END           AS activity,
    0.02 * Balance + 50 * NumOfProducts                                      AS annual_revenue
FROM customers;
"""


def build_database() -> sqlite3.Connection:
    df = pd.read_csv(ROOT / "data" / "churn.csv").drop_duplicates(subset="CustomerId")
    conn = sqlite3.connect(DB_PATH)
    df.to_sql("customers", conn, if_exists="replace", index=False)
    conn.executescript(VIEW_SQL)
    print(f"Loaded {len(df):,} customers into {DB_PATH.name}\n")
    return conn


def parse_queries(sql_text: str) -> dict:
    blocks = re.split(r"^-- name:\s*", sql_text, flags=re.MULTILINE)[1:]
    queries = {}
    for block in blocks:
        name, _, rest = block.partition("\n")
        question = re.search(r"^-- question:\s*(.*)$", rest, flags=re.MULTILINE)
        sql = "\n".join(l for l in rest.splitlines() if not l.startswith("--")).strip()
        queries[name.strip()] = (question.group(1) if question else "", sql)
    return queries


def main():
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    conn = build_database()
    for name, (question, sql) in parse_queries(SQL_TEXT).items():
        result = pd.read_sql_query(sql, conn)
        result.to_csv(OUT_DIR / f"{name}.csv", index=False)
        print(f"=== {name} ===\n{question}")
        print(result.to_string(index=False), "\n")
    conn.close()

main()

## 3. Cleaning, features, model and simulation code
This cell defines the reusable functions (cleaning, feature engineering, model, simulation). Nothing runs yet.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
DATA_PATH = ROOT / "data" / "churn.csv"

# ASSUMPTIONS - revenue per customer per year. Keep in sync with run_sql.py.
NIM_RATE = 0.02          # net interest margin earned on the customer's balance
FEE_PER_PRODUCT = 50     # annual fee / interchange income per product held

NUMERIC = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "HasCrCard",
           "IsActiveMember", "EstimatedSalary"]
CATEGORICAL = ["Geography", "Gender"]


# ----------------------------------------------------------------------------------
# 1. CLEANING + FEATURES
# ----------------------------------------------------------------------------------
def load_and_clean(path: Path = DATA_PATH):
    df = pd.read_csv(path)
    report = {"rows_raw": len(df)}
    df = df.drop_duplicates(subset="CustomerId").copy()
    report["duplicates_removed"] = report["rows_raw"] - len(df)
    report["missing_values"] = int(df[NUMERIC + CATEGORICAL + ["Exited"]].isna().sum().sum())
    df = df.dropna(subset=NUMERIC + CATEGORICAL + ["Exited"])
    report["rows_clean"] = len(df)

    df["is_churn"] = df["Exited"].astype(int)
    df["annual_revenue"] = NIM_RATE * df["Balance"] + FEE_PER_PRODUCT * df["NumOfProducts"]
    df["age_band"] = pd.cut(df["Age"], [0, 29, 39, 49, 59, 120],
                            labels=["<30", "30-39", "40-49", "50-59", "60+"]).astype(str)
    df["balance_band"] = np.select([df["Balance"] == 0, df["Balance"] < 100_000],
                                   ["No balance", "<100k"], default="100k+")
    df["tenure_band"] = pd.cut(df["Tenure"], [-1, 2, 5, 8, 10],
                               labels=["0-2 yrs", "3-5 yrs", "6-8 yrs", "9-10 yrs"]).astype(str)
    df["credit_band"] = pd.cut(df["CreditScore"], [0, 579, 669, 739, 900],
                               labels=["Poor", "Fair", "Good", "Very good"]).astype(str)
    df["activity"] = np.where(df["IsActiveMember"] == 1, "Active", "Inactive")
    df["products_band"] = np.where(df["NumOfProducts"] >= 3, "3+", df["NumOfProducts"].astype(str))
    # Transparent, rule-based business segment (balance tier x engagement)
    df["segment"] = df["balance_band"] + " | " + df["activity"]
    return df.reset_index(drop=True), report


# ----------------------------------------------------------------------------------
# 2. CHURN MODEL
# ----------------------------------------------------------------------------------
def _preprocessor():
    return ColumnTransformer([
        ("num", StandardScaler(), NUMERIC),
        ("cat", OneHotEncoder(drop="first", sparse_output=False), CATEGORICAL),
    ])


def fit_and_score(df: pd.DataFrame):
    """Compare logistic regression vs random forest using 5-fold cross-validation.

    Every customer's churn probability is an OUT-OF-FOLD prediction: the model that scored
    a customer never saw that customer during training. This avoids over-optimistic results.
    """
    X, y = df[NUMERIC + CATEGORICAL], df["is_churn"]
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    models = {
        "Logistic regression": Pipeline([("prep", _preprocessor()),
                                         ("clf", LogisticRegression(max_iter=1000))]),
        "Random forest": Pipeline([("prep", _preprocessor()),
                                   ("clf", RandomForestClassifier(n_estimators=300, max_depth=8,
                                                                  min_samples_leaf=20, random_state=42,
                                                                  n_jobs=-1))]),
    }
    oof = {name: cross_val_predict(m, X, y, cv=cv, method="predict_proba")[:, 1]
           for name, m in models.items()}
    aucs = {name: float(roc_auc_score(y, p)) for name, p in oof.items()}
    best = max(aucs, key=aucs.get)

    final = models[best].fit(X, y)
    names = [n.split("__", 1)[1] for n in final.named_steps["prep"].get_feature_names_out()]
    clf = final.named_steps["clf"]
    imp = clf.feature_importances_ if hasattr(clf, "feature_importances_") else np.abs(clf.coef_[0])
    importance = (pd.DataFrame({"feature": names, "importance": imp / imp.sum()})
                  .sort_values("importance", ascending=False).reset_index(drop=True))

    scored = df.copy()
    scored["churn_prob"] = oof[best]
    rank = scored["churn_prob"].rank(method="first")
    scored["risk_decile"] = pd.qcut(rank, 10, labels=range(1, 11)).astype(int)    # 10 = most likely to churn
    scored["risk_tier"] = pd.cut(scored["churn_prob"], [-0.001, 0.15, 0.40, 1.0],
                                 labels=["Low", "Medium", "High"]).astype(str)
    scored["expected_revenue_loss"] = scored["churn_prob"] * scored["annual_revenue"]
    return scored, {"aucs": aucs, "best_model": best, "importance": importance}


# ----------------------------------------------------------------------------------
# 3. UNSUPERVISED SEGMENTATION (K-Means)
# ----------------------------------------------------------------------------------
def kmeans_segments(df: pd.DataFrame, k: int = 4):
    feats = ["Tenure", "NumOfProducts", "Balance", "IsActiveMember", "Age"]
    Z = StandardScaler().fit_transform(df[feats])
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Z)
    out = df.copy()
    out["cluster"] = km.labels_
    profile = (out.groupby("cluster").agg(
        customers=("CustomerId", "count"), avg_age=("Age", "mean"), avg_tenure=("Tenure", "mean"),
        avg_products=("NumOfProducts", "mean"), avg_balance=("Balance", "mean"),
        pct_active=("IsActiveMember", "mean"), churn_rate=("is_churn", "mean"),
        avg_revenue=("annual_revenue", "mean"), revenue_lost=("annual_revenue",
                                                              lambda s: s[out.loc[s.index, "is_churn"] == 1].sum()))
        .round(3).reset_index())
    return out, profile


# ----------------------------------------------------------------------------------
# 4. RETENTION ROI  (historical replay)
# ----------------------------------------------------------------------------------
def retention_roi(df: pd.DataFrame, target_pct: float, cost_per_customer: float, save_rate: float) -> dict:
    """Target the `target_pct` highest-risk customers with an offer.

    Replay logic: of the customers we would have targeted, some actually left. If the offer
    saves `save_rate` of those leavers, we keep their annual revenue. Every targeted customer costs
    `cost_per_customer` (whether or not they were going to leave).
    """
    n_target = int(round(len(df) * target_pct))
    targeted = df.sort_values("churn_prob", ascending=False).head(n_target)
    leavers = targeted[targeted["is_churn"] == 1]
    revenue_saved = leavers["annual_revenue"].sum() * save_rate
    cost = n_target * cost_per_customer
    return {
        "customers_targeted": n_target,
        "leavers_in_target": int(len(leavers)),
        "share_of_all_churners_reached": len(leavers) / max(df["is_churn"].sum(), 1),
        "revenue_saved": revenue_saved,
        "campaign_cost": cost,
        "net_benefit": revenue_saved - cost,
        "roi": (revenue_saved - cost) / cost if cost else np.nan,
    }

In [ ]:
# make the functions above available as `pl.<name>`, exactly like the scripts do
import types
pl = types.SimpleNamespace(**globals())

## 4. Full analysis: charts, model results, Power BI export files

In [ ]:
import json
from pathlib import Path

import matplotlib

import matplotlib.pyplot as plt
import pandas as pd


OUT = Path.cwd() / "outputs"
CHARTS, PBI = OUT / "charts", OUT / "powerbi"
for d in (CHARTS, PBI):
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold"})
BLUE, RED = "#2F5D8C", "#C0392B"


def churn_bar(df, col, title, fname, order=None):
    rate = df.groupby(col)["is_churn"].mean().mul(100)
    rate = rate.reindex(order) if order else rate.sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(rate.index.astype(str), rate.values, color=BLUE)
    ax.axhline(df["is_churn"].mean() * 100, color=RED, ls="--", lw=1.2, label="Overall churn")
    for i, v in enumerate(rate.values):
        ax.text(i, v + 0.8, f"{v:.1f}%", ha="center", fontsize=8)
    ax.set_title(title)
    ax.set_ylabel("Churn rate (%)")
    ax.legend(frameon=False)
    plt.xticks(rotation=20 if len(rate) > 4 else 0)
    plt.tight_layout()
    fig.savefig(CHARTS / fname)
    plt.close(fig)


def main():
    # ------------------------------------------------------------ 1. CLEAN
    df, report = pl.load_and_clean()
    print("CLEANING REPORT:", report)
    churned = df[df["is_churn"] == 1]
    total_rev, lost_rev = df["annual_revenue"].sum(), churned["annual_revenue"].sum()
    print(f"\nCustomers {len(df):,} | churn rate {df['is_churn'].mean():.1%} | "
          f"annual revenue ${total_rev / 1e6:,.2f}M | revenue lost to churn ${lost_rev / 1e6:,.2f}M "
          f"({lost_rev / total_rev:.1%})")
    print(f"Avg revenue per churned customer ${churned['annual_revenue'].mean():,.0f} "
          f"vs retained ${df[df['is_churn'] == 0]['annual_revenue'].mean():,.0f}")

    # ------------------------------------------------------------ 2. EDA
    churn_bar(df, "Geography", "Churn rate by country", "01_churn_by_geography.png")
    churn_bar(df, "age_band", "Churn rate by age band", "02_churn_by_age.png", ["<30", "30-39", "40-49", "50-59", "60+"])
    churn_bar(df, "products_band", "Churn rate by number of products", "03_churn_by_products.png", ["1", "2", "3+"])
    churn_bar(df, "activity", "Churn rate: active vs inactive members", "04_churn_by_activity.png")
    churn_bar(df, "segment", "Churn rate by customer segment", "05_churn_by_segment.png")

    seg = (df.groupby("segment").agg(customers=("CustomerId", "count"), churn_rate=("is_churn", "mean"),
                                     revenue=("annual_revenue", "sum"))
           .assign(revenue_lost=lambda s: df[df.is_churn == 1].groupby("segment")["annual_revenue"].sum())
           .fillna(0).sort_values("revenue_lost", ascending=False).reset_index())
    seg["share_of_loss"] = seg["revenue_lost"] / seg["revenue_lost"].sum()
    print("\nREVENUE LEAKAGE BY SEGMENT")
    print(seg.assign(churn_rate=lambda s: s.churn_rate.round(3), revenue_lost=lambda s: s.revenue_lost.round(0),
                     share_of_loss=lambda s: s.share_of_loss.round(3))
          [["segment", "customers", "churn_rate", "revenue_lost", "share_of_loss"]].to_string(index=False))

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.barh(seg["segment"][::-1], seg["revenue_lost"][::-1] / 1e3, color=RED)
    ax.set_xlabel("Annual revenue lost to churn ($K)")
    ax.set_title("Revenue leakage by segment")
    plt.tight_layout()
    fig.savefig(CHARTS / "06_revenue_leakage_by_segment.png")
    plt.close(fig)

    # ------------------------------------------------------------ 3. MODEL
    scored, info = pl.fit_and_score(df)
    print("\nCHURN MODEL (5-fold cross-validated AUC)")
    for name, auc in info["aucs"].items():
        print(f"  {name:<22}{auc:.3f}")
    print(f"  -> using: {info['best_model']}")
    print("\n  Top drivers (feature importance):")
    print(info["importance"].head(6).round(3).to_string(index=False))

    dec = (scored.groupby("risk_decile").agg(customers=("CustomerId", "count"),
                                             avg_churn_prob=("churn_prob", "mean"),
                                             actual_churn_rate=("is_churn", "mean"),
                                             churners=("is_churn", "sum"),
                                             avg_revenue=("annual_revenue", "mean"),
                                             total_revenue=("annual_revenue", "sum"),
                                             revenue_lost=("annual_revenue",
                                                           lambda s: s[scored.loc[s.index, "is_churn"] == 1].sum()))
           .reset_index())
    top2 = dec[dec.risk_decile >= 9]
    print(f"\n  Top 20% riskiest customers hold {top2.churners.sum() / dec.churners.sum():.1%} of all churners "
          f"(random targeting would reach 20%)  -> LIFT {top2.churners.sum() / dec.churners.sum() / 0.2:.2f}x")

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(dec["risk_decile"], dec["actual_churn_rate"] * 100, color=BLUE)
    ax.set_xticks(range(1, 11))
    ax.set_xlabel("Risk decile (10 = model says most likely to churn)")
    ax.set_ylabel("Actual churn rate (%)")
    ax.set_title("Model works: churn rate climbs with risk decile")
    plt.tight_layout()
    fig.savefig(CHARTS / "07_churn_by_risk_decile.png")
    plt.close(fig)

    imp = info["importance"].head(8).iloc[::-1]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.barh(imp["feature"], imp["importance"] * 100, color=BLUE)
    ax.set_xlabel("Importance (%)")
    ax.set_title("What drives churn?")
    plt.tight_layout()
    fig.savefig(CHARTS / "08_feature_importance.png")
    plt.close(fig)

    # ------------------------------------------------------------ 4. K-MEANS
    clustered, profile = pl.kmeans_segments(scored)
    print("\nK-MEANS CUSTOMER CLUSTERS (k=4)")
    print(profile.to_string(index=False))

    # ------------------------------------------------------------ 5. RETENTION ROI
    print("\nRETENTION CAMPAIGN ROI  (assumptions: offer cost $50/customer, offer saves 30% of would-be leavers)")
    rows = []
    prev_net, prev_cost = 0, 0
    for pct in [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]:
        r = pl.retention_roi(scored, pct, 50, 0.30)
        r["target_pct"] = pct
        r["marginal_net_benefit"] = r["net_benefit"] - prev_net
        r["marginal_roi"] = r["marginal_net_benefit"] / (r["campaign_cost"] - prev_cost)
        prev_net, prev_cost = r["net_benefit"], r["campaign_cost"]
        rows.append(r)
    roi = pd.DataFrame(rows)
    print(roi[["target_pct", "customers_targeted", "share_of_all_churners_reached", "revenue_saved",
               "campaign_cost", "net_benefit", "roi", "marginal_roi"]].round(2).to_string(index=False))
    # Rule: keep widening the campaign while each EXTRA dollar spent returns more than $1 of net benefit
    best = roi[roi["marginal_roi"] >= 1].iloc[-1]
    print(f"\n  Recommended volume: target top {best.target_pct:.0%} "
          f"(last tranche where marginal ROI >= 1.0x) -> net benefit ${best.net_benefit:,.0f}, "
          f"reaches {best.share_of_all_churners_reached:.0%} of churners")

    # ------------------------------------------------------------ 6. EXPORTS
    scored.to_csv(PBI / "customers_scored.csv", index=False)
    clustered[["CustomerId", "cluster"]].to_csv(PBI / "customer_clusters.csv", index=False)
    profile.to_csv(PBI / "cluster_profile.csv", index=False)
    seg.to_csv(PBI / "segment_leakage.csv", index=False)
    dec.to_csv(PBI / "risk_deciles.csv", index=False)
    info["importance"].to_csv(PBI / "feature_importance.csv", index=False)
    roi.to_csv(PBI / "retention_roi.csv", index=False)
    (OUT / "metrics.json").write_text(json.dumps({
        "customers": len(df), "churn_rate": df["is_churn"].mean(), "revenue_total": total_rev,
        "revenue_lost": lost_rev, "revenue_lost_pct": lost_rev / total_rev,
        "model_aucs": info["aucs"], "best_model": info["best_model"],
        "top20pct_reach": float(top2.churners.sum() / dec.churners.sum()),
        "recommended_target_pct": float(best.target_pct), "recommended_net_benefit": float(best.net_benefit),
        "recommended_reach": float(best.share_of_all_churners_reached), "recommended_roi": float(best.roi),
    }, indent=2, default=float))
    print(f"\nSaved charts -> {CHARTS}\nSaved Power BI files -> {PBI}")

main()

### Charts produced
Churn by country / age / products / activity / segment, revenue leakage, risk deciles and feature importance.

In [ ]:
from pathlib import Path
try:
    from IPython.display import Image, display
    for f in sorted(Path('outputs/charts').glob('*.png')):
        print(f.name)
        display(Image(filename=str(f), width=520))
except ImportError:
    print('Charts saved in outputs/charts/')

## 5. Excel what-if workbook
Builds an Excel file made of live formulas. Open it in Excel and change the yellow cells.

In [ ]:
from pathlib import Path

from openpyxl import Workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.worksheet.datavalidation import DataValidation


OUT = Path.cwd() / "outputs" / "Retention_ROI_Calculator.xlsx"
F = "Arial"
HDR_FILL = PatternFill("solid", fgColor="1F3A5F")
INPUT_FILL = PatternFill("solid", fgColor="FFFF00")
BAND_FILL = PatternFill("solid", fgColor="EAF1F8")
thin = Side(style="thin", color="BBBBBB")
BOX = Border(left=thin, right=thin, top=thin, bottom=thin)
USD, PCT, INT, MULT = '$#,##0;($#,##0);-', '0.0%', '#,##0', '0.0"x"'


def header(ws, row, labels):
    for c, h in enumerate(labels, start=1):
        cell = ws.cell(row=row, column=c, value=h)
        cell.font = Font(name=F, bold=True, color="FFFFFF")
        cell.fill = HDR_FILL
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = BOX
    ws.row_dimensions[row].height = 32


def put(ws, ref, value, fmt=None, bold=False, color="000000", fill=None):
    c = ws[ref]
    c.value = value
    c.font = Font(name=F, bold=bold, color=color)
    if fmt:
        c.number_format = fmt
    if fill:
        c.fill = fill
    c.border = BOX
    return c


def main():
    df, _ = pl.load_and_clean()
    scored, _ = pl.fit_and_score(df)
    dec = (scored.groupby("risk_decile").agg(
        customers=("CustomerId", "count"), churners=("is_churn", "sum"),
        revenue=("annual_revenue", "sum")).reset_index())
    dec["revenue_lost"] = scored[scored.is_churn == 1].groupby("risk_decile")["annual_revenue"].sum().values
    seg = (scored.groupby("segment").agg(customers=("CustomerId", "count"), churners=("is_churn", "sum"))
           .reset_index())
    seg["revenue_lost"] = scored[scored.is_churn == 1].groupby("segment")["annual_revenue"].sum().reindex(seg.segment).fillna(0).values
    seg = seg.sort_values("revenue_lost", ascending=False).reset_index(drop=True)

    wb = Workbook()

    # ---------------------------------------------------------------- Instructions
    ws = wb.active
    ws.title = "Instructions"
    ws.sheet_view.showGridLines = False
    ws["A1"].value = "Retention Campaign ROI Calculator"
    ws["A1"].font = Font(name=F, size=16, bold=True)
    lines = [
        "PURPOSE: Decide how many at-risk customers to target with a retention offer, and whether it pays off.",
        "",
        "HOW TO USE",
        "  1. Go to 'ROI_Calculator'.",
        "  2. Change the YELLOW cells: how many risk deciles to target, offer cost per customer, and the offer's save rate.",
        "  3. Read net benefit, ROI and break-even save rate. The sensitivity table shows many scenarios at once.",
        "",
        "COLOUR LEGEND",
        "  Blue text = data from the Python churn model   |   Black = formulas   |   Yellow fill = inputs you can change",
        "",
        "KEY ASSUMPTIONS (editable in the Python code):",
        "  - Annual revenue per customer = 2% net interest margin x balance + $50 fee income per product held.",
        "  - Customers are ranked into 10 risk deciles by the model's churn probability (decile 10 = most likely to leave).",
        "  - 'Save rate' = share of would-be leavers the offer keeps. Benefit counted = 1 year of revenue only (conservative).",
        "  - The offer is paid for every targeted customer, including those who would have stayed anyway.",
        "",
        "Example: target 3 deciles, $50 per customer, 30% save rate -> see the results update on 'ROI_Calculator'.",
    ]
    for i, t in enumerate(lines, start=3):
        ws.cell(row=i, column=1, value=t).font = Font(name=F, bold=t.isupper() or t.startswith(("HOW", "COLOUR", "KEY", "PURPOSE")))
    ws.column_dimensions["A"].width = 125

    # ---------------------------------------------------------------- Risk_Deciles
    d = wb.create_sheet("Risk_Deciles")
    d["A1"].value = "Customers by model risk decile (10 = most likely to churn)"
    d["A1"].font = Font(name=F, size=13, bold=True)
    header(d, 4, ["Risk Decile", "Customers", "Churners (actual)", "Annual Revenue ($)",
                  "Revenue Lost to Churn ($)", "Churn Rate", "Share of All Churners"])
    for i, row in dec.iterrows():
        r = 5 + i
        put(d, f"A{r}", int(row.risk_decile), INT, bold=True, color="0000FF")
        put(d, f"B{r}", int(row.customers), INT, color="0000FF")
        put(d, f"C{r}", int(row.churners), INT, color="0000FF")
        put(d, f"D{r}", float(row.revenue), USD, color="0000FF")
        put(d, f"E{r}", float(row.revenue_lost), USD, color="0000FF")
        put(d, f"F{r}", f"=IFERROR(C{r}/B{r},0)", PCT)
        put(d, f"G{r}", f"=C{r}/$C$15", PCT)
    put(d, "A15", "Total", bold=True, fill=BAND_FILL)
    for col, fmt in zip("BCDE", [INT, INT, USD, USD]):
        put(d, f"{col}15", f"=SUM({col}5:{col}14)", fmt, bold=True, fill=BAND_FILL)
    put(d, "F15", "=IFERROR(C15/B15,0)", PCT, bold=True, fill=BAND_FILL)
    put(d, "G15", "=SUM(G5:G14)", PCT, bold=True, fill=BAND_FILL)
    for col, w in zip("ABCDEFG", [14, 14, 16, 20, 22, 14, 18]):
        d.column_dimensions[col].width = w
    ch = BarChart()
    ch.title = "Churn rate by risk decile"
    ch.add_data(Reference(d, min_col=6, min_row=4, max_row=14), titles_from_data=True)
    ch.set_categories(Reference(d, min_col=1, min_row=5, max_row=14))
    ch.y_axis.numFmt = "0%"
    ch.legend = None
    ch.height, ch.width = 7.5, 14
    d.add_chart(ch, "A18")

    # ---------------------------------------------------------------- Segment_Leakage
    s = wb.create_sheet("Segment_Leakage")
    s["A1"].value = "Revenue leakage by customer segment (balance tier | activity)"
    s["A1"].font = Font(name=F, size=13, bold=True)
    header(s, 4, ["Segment", "Customers", "Churners", "Churn Rate", "Revenue Lost ($)",
                  "Share of Total Loss", "Cumulative Share"])
    n = len(seg)
    for i, row in seg.iterrows():
        r = 5 + i
        put(s, f"A{r}", row.segment, bold=True)
        put(s, f"B{r}", int(row.customers), INT, color="0000FF")
        put(s, f"C{r}", int(row.churners), INT, color="0000FF")
        put(s, f"D{r}", f"=IFERROR(C{r}/B{r},0)", PCT)
        put(s, f"E{r}", float(row.revenue_lost), USD, color="0000FF")
        put(s, f"F{r}", f"=E{r}/$E${5 + n}", PCT)
        put(s, f"G{r}", f"=SUM($F$5:F{r})", PCT)
    tr = 5 + n
    put(s, f"A{tr}", "Total", bold=True, fill=BAND_FILL)
    for col, fmt in zip("BCE", [INT, INT, USD]):
        put(s, f"{col}{tr}", f"=SUM({col}5:{col}{tr - 1})", fmt, bold=True, fill=BAND_FILL)
    put(s, f"D{tr}", f"=IFERROR(C{tr}/B{tr},0)", PCT, bold=True, fill=BAND_FILL)
    put(s, f"F{tr}", f"=SUM(F5:F{tr - 1})", PCT, bold=True, fill=BAND_FILL)
    for col, w in zip("ABCDEFG", [26, 12, 12, 12, 18, 18, 16]):
        s.column_dimensions[col].width = w
    ch2 = BarChart()
    ch2.type = "bar"
    ch2.title = "Revenue lost to churn by segment ($)"
    ch2.add_data(Reference(s, min_col=5, min_row=4, max_row=tr - 1), titles_from_data=True)
    ch2.set_categories(Reference(s, min_col=1, min_row=5, max_row=tr - 1))
    ch2.legend = None
    ch2.height, ch2.width = 8, 15
    s.add_chart(ch2, "A15")

    # ---------------------------------------------------------------- ROI_Calculator
    c = wb.create_sheet("ROI_Calculator", 1)
    c.sheet_view.showGridLines = False
    c["A1"].value = "Retention Campaign ROI Calculator"
    c["A1"].font = Font(name=F, size=15, bold=True)
    c["A2"].value = "Change the yellow cells. Everything else recalculates."
    c["A2"].font = Font(name=F, italic=True)

    c["A4"].value = "INPUTS"
    c["A4"].font = Font(name=F, bold=True, size=12)
    put(c, "A5", "Risk deciles to target (1-10, riskiest first)", bold=True)
    put(c, "B5", 3, INT, color="0000FF", bold=True, fill=INPUT_FILL)
    put(c, "A6", "Offer cost per targeted customer ($)", bold=True)
    put(c, "B6", 50, USD, color="0000FF", bold=True, fill=INPUT_FILL)
    put(c, "A7", "Save rate (share of would-be leavers kept)", bold=True)
    put(c, "B7", 0.30, PCT, color="0000FF", bold=True, fill=INPUT_FILL)
    dv = DataValidation(type="whole", operator="between", formula1="1", formula2="10")
    c.add_data_validation(dv)
    dv.add("B5")
    dv2 = DataValidation(type="decimal", operator="between", formula1="0", formula2="1")
    c.add_data_validation(dv2)
    dv2.add("B7")

    c["A9"].value = "RESULTS"
    c["A9"].font = Font(name=F, bold=True, size=12)
    crit = '">="&(11-$B$5)'
    results = [
        ("Customers targeted", f"=SUMIFS(Risk_Deciles!$B$5:$B$14,Risk_Deciles!$A$5:$A$14,{crit})", INT),
        ("% of customer base targeted", "=B10/Risk_Deciles!$B$15", PCT),
        ("Would-be leavers inside the target group", f"=SUMIFS(Risk_Deciles!$C$5:$C$14,Risk_Deciles!$A$5:$A$14,{crit})", INT),
        ("% of ALL churners reached", "=B12/Risk_Deciles!$C$15", PCT),
        ("Annual revenue of those leavers ($)", f"=SUMIFS(Risk_Deciles!$E$5:$E$14,Risk_Deciles!$A$5:$A$14,{crit})", USD),
        ("Revenue saved by the offer ($)", "=B14*B7", USD),
        ("Campaign cost ($)", "=B10*B6", USD),
        ("NET BENEFIT ($)", "=B15-B16", USD),
        ("ROI (net benefit / cost)", "=IFERROR(B17/B16,0)", MULT),
        ("Break-even save rate", "=IFERROR(B16/B14,0)", PCT),
        ("Lift vs random targeting", "=IFERROR(B13/B11,0)", MULT),
    ]
    for i, (label, formula, fmt) in enumerate(results):
        r = 10 + i
        bold = label.startswith("NET")
        put(c, f"A{r}", label, bold=bold, fill=BAND_FILL if bold else None)
        put(c, f"B{r}", formula, fmt, bold=bold, fill=BAND_FILL if bold else None)

    c["A23"].value = "SENSITIVITY: Net benefit ($) at the targeting level above - save rate (down) vs cost per customer (across)"
    c["A23"].font = Font(name=F, bold=True, size=12)
    put(c, "A24", "Save rate  \\  Cost per customer", bold=True, fill=BAND_FILL)
    costs = [25, 50, 75, 100, 150]
    saves = [0.10, 0.20, 0.30, 0.40, 0.50]
    for j, cost in enumerate(costs):
        col = "BCDEF"[j]
        put(c, f"{col}24", cost, USD, color="0000FF", bold=True, fill=BAND_FILL)
    for i, sv in enumerate(saves):
        r = 25 + i
        put(c, f"A{r}", sv, PCT, color="0000FF", bold=True, fill=BAND_FILL)
        for j in range(len(costs)):
            col = "BCDEF"[j]
            put(c, f"{col}{r}", f"=$B$14*$A{r}-$B$10*{col}$24", USD)
    c["A31"].value = "Values in (brackets) are negative = the campaign loses money at that combination."
    c["A31"].font = Font(name=F, italic=True)
    c.column_dimensions["A"].width = 52
    for col in "BCDEF":
        c.column_dimensions[col].width = 16

    wb.save(OUT)
    print(f"Saved {OUT}")

main()

## 6. Key findings (10,000 customers)

- **18.3%** of customers churned, taking **$3.2M (20.5%)** of annual revenue (revenue = 2% of balance + $50 per product: an assumption).
- Inactive customers with 100k+ balances are **22% of customers but ~51% of revenue lost**.
- Products held is U-shaped: **1 product 25%, 2 products 7%, 3 products 71%**.
- Random forest **AUC 0.80** vs logistic regression **0.70** (5-fold cross-validation). Top 20% by risk contain **52%** of churners (2.6x lift).
- Targeting the top **30%** by risk: reaches 66% of churners, net benefit about **$542K, ROI 3.6x** (assumes $50 per offer, 30% save rate). Beyond 30% each extra dollar returns less than $1.

**Next:** build the Power BI dashboard from `outputs/powerbi/*.csv` using `POWERBI_GUIDE.md`; deploy `app.py` on Streamlit Community Cloud.